# The Numerical Simulation

Before training the PICON, we will need a training and testing data set from numerical simulation. The numerical code is "hw2d", which is available to download on github https://github.com/the-rccg/hw2d . The hw2d code starts from a random initial plasma state, then evolve the plasma state over time steps according to the Hasegawa-Wakatain (HW) equation system. 

In this notebook, we prepare the numerical simulation code in batch submission system. Apart from the different initial states sampled from random number generator, each numerical simulation instance has different PDE parameters: c1, k0, kappa, and poisson_bracket_coefficicent (pbc). The value of c1 is sampled between 0.9 and 1.1, the value of k0 is sampled between 0.55 and 0.65, the value of kappa be sampled between 0.9 and 1.1, the value of pbc is sampled between 0.9 and 1. These values are input into the hw2d program keyword arguments, and saved into a numpy array. 

In [1]:
import os
import h5py
import glob
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
comOneTemp='''python -m hw2d \
--step_size=0.005 --end_time=200 --grid_pts=128 --N=3 --nu=5.0e-10 \
--c1=[c1Value] --k0=[k0Value] --kappa_coeff=[kappaCoeff] --poisson_bracket_coeff=[poissonBracketCoeff] --output_path="[outputPath]" \
--buffer_length=4000 --snaps=1 --downsample_factor=1 --movie=0 --debug=0
'''
for i in range(8):
    slurmGo=slurmTemp+'tamulauncher Commandor_'+str(i)+'.in \n'
    with open('FiveHWRun_'+str(i)+'.slurm','w') as writter:
        writter.write(slurmGo)
    c1List=[]
    k0List=[]
    kappaList=[]
    pbcList=[]
    commandor=''
    for j in range(80):
        c1One=np.random.random()*0.2+0.9
        k0One=np.random.random()*0.1+0.55
        kappaOne=np.random.random()*0.2+0.9
        pbcOne=np.random.random()*0.1+0.9
        outputPath='FiveHWRunner/storeField_'+str(i)+'_'+str(j)+'.h5'
        
        comOneGo=comOneTemp.replace('[c1Value]',str(c1One))
        comOneGo=comOneGo.replace('[k0Value]',str(k0One))
        comOneGo=comOneGo.replace('[kappaCoeff]',str(kappaOne))
        comOneGo=comOneGo.replace('[poissonBracketCoeff]',str(pbcOne))
        comOneGo=comOneGo.replace('[outputPath]',outputPath)
        commandor=commandor+comOneGo
        
        c1List.append(c1One)
        k0List.append(k0One)
        kappaList.append(kappaOne)
        pbcList.append(pbcOne)
    with open('Commandor_'+str(i)+'.in','w') as writter:
        writter.write(commandor)
    np.save('FiveHWRunner/c1List_'+str(i)+'.npy',c1List)
    np.save('FiveHWRunner/k0List_'+str(i)+'.npy',k0List)
    np.save('FiveHWRunner/kappaList_'+str(i)+'.npy',kappaList)
    np.save('FiveHWRunner/pbcList_'+str(i)+'.npy',pbcList)
    

# Results

After running this notebook, a series of running scripts like "Commandor_0.in" are generated, you could use the following command to start the hw2d numerical simulation, the numerical simulation results will be stored in the "FiveHWRunner/" directory. The PDE parameters are also stored in the "FiveHWRunner/" directory, with the file names "c1List_\*.npy", "k0List_\*.npy", "kappaList_\*.npy", and "pbcList_\*.npy". 

In [ ]:
!./Commandor_0.in 